# ReAct Agent Basics (Chapter 8)

This notebook accompanies Chapter 8 §8.2.1 and §8.2.2 of *Context Engineering with DSPy*. You will define three plain Python tools, hand them to `dspy.ReAct`, and run an autonomous tool-using agent end-to-end. The second half of the notebook shows how the same `ReAct` module scales to a multi-output customer-service signature and how to construct a `dspy.Tool` explicitly when you need more control over its schema.

**Required environment variable**

- `OPENAI_API_KEY` — used by the default `openai/gpt-5-mini` LM.

Create a `.env` file in the repo root (or export the variable in your shell) before running the setup cell.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5-mini"))

## Defining tools

A tool in DSPy is any Python callable. `dspy.ReAct` inspects the function name, docstring, and type hints to build the schema the LLM sees. Below are three minimal tools you can extend later with real retrieval, database lookups, and safe arithmetic.

In [ ]:
def search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    # Replace this stub with a call to your retriever or vector store.
    results = f"[stub] documentation results for: {query}"
    return results


def get_order_status(order_id: str) -> str:
    """Look up the current status of a customer order by its ID."""
    # Replace this stub with a real database lookup.
    status = f"Order {order_id}: shipped, arriving Tuesday."
    return status


def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))

## A first ReAct agent

Pass the tools straight to `dspy.ReAct` with a signature describing the task. The agent runs a think/act/observe loop until it calls the built-in `finish` tool or hits `max_iters`.

In [ ]:
agent = dspy.ReAct(
    "question -> answer",
    tools=[search_knowledge_base, get_order_status, calculate],
    max_iters=10,
)

result = agent(question="What's the status of order #12345?")
print(result.answer)

## Signature-polymorphic ReAct

`dspy.ReAct` is signature-polymorphic. You can declare typed multi-output signatures and DSPy will populate them in the final extraction step. Below is a customer-service agent that returns both a response string and an `escalation_needed` boolean.

In [ ]:
# Stub tools so the example is runnable without extra infrastructure.
def search_kb(query: str) -> str:
    """Search the internal knowledge base for help articles."""
    return f"[stub] kb hit for: {query}"


def get_order(order_id: str) -> str:
    """Fetch the order record for a given order ID."""
    return f"[stub] order {order_id} found."


def get_account(customer_id: str) -> str:
    """Fetch the customer account record for a given customer ID."""
    return f"[stub] account {customer_id} found."


def file_ticket(summary: str) -> str:
    """File a support ticket with a short summary."""
    return f"[stub] ticket filed: {summary}"


# Customer service agent
service_agent = dspy.ReAct(
    "customer_query, customer_id -> response, escalation_needed: bool",
    tools=[search_kb, get_order, get_account, file_ticket],
    max_iters=8,
)

service_result = service_agent(
    customer_query="My order #12345 never arrived, can you help?",
    customer_id="C-001",
)
print(service_result.response)
print("escalation_needed:", service_result.escalation_needed)

## Building a `dspy.Tool` explicitly

Most of the time you can let DSPy infer everything from the function signature and docstring. When you want full control — custom name, description, per-argument hints — wrap the callable in `dspy.Tool` yourself and pass it into the `tools` list.

In [ ]:
def my_function(query: str) -> str:
    """Default docstring; will be overridden by the explicit dspy.Tool config below."""
    return f"[stub] docs lookup for: {query}"


tool = dspy.Tool(
    func=my_function,
    name="search_docs",
    desc="Search internal documentation by keyword",
    arg_desc={"query": "The search query to run against the docs"},
)

print(tool)